# 📊 Guía: Evaluación de Modelos de Clasificación Binaria

## ¿Para qué sirve este código?

Cuando entrenas un modelo que predice **dos categorías** (Sí/No, 0/1, Sobrevivió/No sobrevivió), necesitas saber **qué tan bien está prediciendo**. Este código te da:

- Una **matriz de confusión** → visual de aciertos y errores
- Tres **métricas numéricas** → qué tan bueno es el modelo

---

## ✅ ¿Cuándo usarlo? Solo con clasificación BINARIA

| Ejemplo | Clase 0 | Clase 1 |
|---|---|---|
| Titanic | No sobrevivió | Sobrevivió |
| Médico | Sano | Enfermo |
| Banco | No fraude | Fraude |
| Email | No spam | Spam |

## ❌ ¿Cuándo NO usarlo?
- Predices precios o números continuos → **eso es regresión**
- Tienes 3 o más categorías → **clasificación multiclase**

---

## Las 3 Métricas — Analogía del detector de incendios

| Métrica | Pregunta que responde | Cuándo importa más |
|---|---|---|
| **Accuracy** | ¿Cuántas veces acertó en total? | Clases balanceadas |
| **Precision** | Cuando dijo "SÍ", ¿tenía razón? | Evitar falsas alarmas (spam) |
| **Recall** | De todos los "SÍ" reales, ¿cuántos detectó? | No perderse casos reales (medicina) |

---
# 🛠️ Implementación Paso a Paso

## PASO 1: Importar librerías

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('✅ Librerías importadas correctamente')

## PASO 2: Las dos funciones (copiar tal cual)

> ⚠️ Solo necesitas entender **qué reciben** y **qué devuelven**. No necesitas modificarlas.

In [ ]:
def compute_metrics(y_true, y_pred):
    """
    Recibe:  y_true → valores REALES   (ej: [1, 0, 1, 1, 0])
             y_pred → valores PREDICHOS (ej: [1, 0, 0, 1, 0])
    Devuelve: DataFrame con accuracy, precision y recall
    """
    accuracy  = round(100.0 * accuracy_score(y_true, y_pred), 2)
    report    = classification_report(y_true, y_pred, output_dict=True)
    precision = round(report['1']['precision'], 3)
    recall    = round(report['1']['recall'], 3)

    metrics_df = pd.DataFrame({
        'accuracy':  [accuracy],
        'precision': [precision],
        'recall':    [recall]
    }, index=['metricas'])
    return metrics_df


def plot_confusion_matrix_and_reports(y_true, y_pred, title='Matriz de Confusión', cmap=plt.cm.Blues):
    """
    Recibe:  y_true → valores REALES
             y_pred → valores PREDICHOS
             title  → texto del título del gráfico (opcional)
    Devuelve: muestra un gráfico con matriz de confusión + métricas
    """
    metrics_df  = compute_metrics(y_true, y_pred)
    conf_matrix = confusion_matrix(y_true, y_pred)

    def plot_confusion_matrix(cm, metrics):
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))

        sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                    xticklabels=['No Sobr.', 'Sobr.'],
                    yticklabels=['No Sobr.', 'Sobr.'], ax=axes[0])
        axes[0].set_title(title)
        axes[0].set_xlabel('Predicción')
        axes[0].set_ylabel('Real')

        metrics_text  = f"Accuracy:  {metrics['accuracy']/100:.3f} - ({metrics['accuracy']:.1f}%)\n"
        metrics_text += f"Precision: {metrics['precision']:.3f} - ({metrics['precision']*100:.1f}%)\n"
        metrics_text += f"Recall:    {metrics['recall']:.3f} - ({metrics['recall']*100:.1f}%)\n"

        axes[1].axis('off')
        axes[1].text(0.5, 0.5, metrics_text, horizontalalignment='center',
                     verticalalignment='center', fontsize=12)

        plt.tight_layout()
        plt.show()

    plot_confusion_matrix(conf_matrix, metrics_df.loc['metricas'])


print('✅ Funciones definidas correctamente')

---
## PASO 3: Cargar datos y entrenar un modelo

> Usamos el dataset del Titanic como ejemplo. Aquí es donde **cambiarías tus propios datos**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# --- Cargar datos del Titanic ---
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df  = pd.read_csv(url)

# --- Preparar features simples ---
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].dropna()
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})  # convertir texto a número

# --- Separar X (features) e y (objetivo) ---
X = df[['Pclass', 'Sex', 'Age', 'Fare']]  # ← lo que el modelo observa
y = df['Survived']                         # ← lo que el modelo predice (0 o 1)

# --- Dividir en entrenamiento (80%) y prueba (20%) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Escalar y entrenar ---
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

modelo = LogisticRegression(random_state=42)
modelo.fit(X_train_scaled, y_train)

# --- Predecir ---
y_pred_train = modelo.predict(X_train_scaled)
y_pred_test  = modelo.predict(X_test_scaled)

print('✅ Modelo entrenado. Datos listos para evaluar.')

---
## PASO 4: Usar las funciones

### Opción A — Solo métricas numéricas

In [ ]:
# Solo quiero ver los números
metricas = compute_metrics(y_test, y_pred_test)
print(metricas)

### Opción B — Gráfico completo (matriz + métricas)

In [ ]:
# Evaluar conjunto de PRUEBA
plot_confusion_matrix_and_reports(
    y_true = y_test,
    y_pred = y_pred_test,
    title  = 'Conjunto de Prueba'
)

In [ ]:
# Evaluar conjunto de ENTRENAMIENTO (para comparar)
plot_confusion_matrix_and_reports(
    y_true = y_train,
    y_pred = y_pred_train,
    title  = 'Conjunto de Entrenamiento'
)

---
## 💡 ¿Cómo leer la Matriz de Confusión?

```
                  PREDICHO
                No Sobr. | Sobr.
         --------|--------|--------
REAL  No Sobr. |   TN   |   FP    ← Falsa alarma
         --------|--------|--------
         Sobr.  |   FN   |   TP
                    ↑
               Se escapó
```

- **TN** (True Negative): Dijo No y era No ✅
- **TP** (True Positive): Dijo Sí y era Sí ✅  
- **FP** (False Positive): Dijo Sí pero era No ❌ → Falsa alarma
- **FN** (False Negative): Dijo No pero era Sí ❌ → El más peligroso en medicina

---
## 🔄 Plantilla para tus propios datos

Solo cambia las 4 líneas marcadas con `# ← CAMBIA ESTO`:

In [ ]:
# ============================================================
# PLANTILLA — copia y adapta para cualquier proyecto
# ============================================================
from sklearn.ensemble import RandomForestClassifier  # ← CAMBIA ESTO por tu modelo

# X = tus_datos[['col1', 'col2', ...]]  # ← CAMBIA ESTO
# y = tus_datos['objetivo']             # ← CAMBIA ESTO (columna con 0s y 1s)

X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.2, random_state=42)

mi_modelo = RandomForestClassifier(random_state=42)  # ← CAMBIA ESTO
mi_modelo.fit(X_train2, y_train2)

y_pred2 = mi_modelo.predict(X_test2)

# Evaluar — esto NO cambia nunca
plot_confusion_matrix_and_reports(y_test2, y_pred2, title='Mi Modelo')